In [117]:
import pandas as pd
import plotly.express as px
import folium
import yaml

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=False)

In [103]:
model = calliope.read_yaml('model.yaml')

[2026-03-16 13:40:13] INFO     Math init | loading pre-defined math.
[2026-03-16 13:40:13] INFO     Math init | loading math files {'storage_inter_cluster', 'milp', 'base', 'spores', 'operate'}.
[2026-03-16 13:40:13] INFO     Model: preprocessing data
[2026-03-16 13:40:13] INFO     Math build | building applied math with ['base'].
[2026-03-16 13:40:13] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 13:40:13] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 13:40:13] INFO     input data `cost_source_use` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 13:40:13] INFO     input data `link_to` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 13:40:13] INFO     input data `link_from` not defined in model math; it will not be available in the optimisation proble

In [104]:
model.inputs

<xarray.Dataset> Size: 10kB
Dimensions:                     (costs: 1, techs: 5, nodes: 3, carriers: 1,
                                 timesteps: 48)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 40B 'X1_to_X2' ... 'supply_gri...
  * carriers                    (carriers) object 8B 'electricity'
  * nodes                       (nodes) object 24B 'X1' 'X2' 'X3'
  * timesteps                   (timesteps) datetime64[ns] 384B 2005-07-01 .....
Data variables: (12/28)
    cost_interest_rate          (costs) float64 8B 0.1
    bigM                        float64 8B 1e+06
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 40B 'transmission' ... 'supply'
    carrier_in                  (nodes, techs, carriers) bool 15B True ... False
    color                       (techs) object 40B '#823739' ... '#C5ABE3'
    ...                          ...
    source_use_equals           (techs, timesteps) float64 2kB nan nan ... nan
    sink_use_equals             (timesteps, techs, nodes) float64 6kB nan ......
    definition_matrix           (nodes, techs, carriers) bool 15B True ... False
    distance                    (techs) float64 40B 0.806 1.074 nan nan nan
    timestep_resolution         (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0
    timestep_weights            (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0

In [105]:
model.inputs.flow_cap_max.to_series().dropna()

techs
X1_to_X2             10000.0
X1_to_X3             10000.0
pv                     250.0
supply_grid_power     2000.0
Name: flow_cap_max, dtype: float64

In [106]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

techs               nodes
demand_electricity  X1         35.271156
                    X2       8796.878622
                    X3       1244.604116
Name: sink_use_equals, dtype: float64

In [107]:
model.build(force=True)
model.solve(solver='gurobi')

[2026-03-16 13:40:13] INFO     Model: backend build starting
[2026-03-16 13:40:14] INFO     Optimisation Model | parameters/lookups | Generated.
[2026-03-16 13:40:14] INFO     Optimisation Model | variables | Generated.
[2026-03-16 13:40:15] INFO     Optimisation Model | global_expressions | Generated.
[2026-03-16 13:40:16] INFO     Optimisation Model | constraints | Generated.
[2026-03-16 13:40:16] INFO     Optimisation Model | piecewise_constraints | Generated.
[2026-03-16 13:40:17] INFO     Optimisation Model | objectives | Generated.
[2026-03-16 13:40:17] INFO     Model: backend build complete
[2026-03-16 13:40:17] INFO     Optimisation model | starting model in base mode.
[2026-03-16 13:40:17] INFO     Backend: solver finished running. Time since start of solving optimisation problem: 0:00:00.128082
[2026-03-16 13:40:17] INFO     Postprocessing: applied zero threshold 1e-10 to model results.
[2026-03-16 13:40:17] INFO     Postprocessing: ended. Time since start of solving optimisa

In [108]:
model.results

<xarray.Dataset> Size: 45kB
Dimensions:                     (nodes: 3, techs: 5, carriers: 1,
                                 timesteps: 48, costs: 1)
Coordinates:
  * techs                       (techs) object 40B 'X1_to_X2' ... 'supply_gri...
  * nodes                       (nodes) object 24B 'X1' 'X2' 'X3'
  * carriers                    (carriers) object 8B 'electricity'
  * timesteps                   (timesteps) datetime64[ns] 384B 2005-07-01 .....
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/21)
    flow_cap                    (nodes, techs, carriers) float64 120B 271.5 ....
    link_flow_cap               (techs) float64 40B 271.5 50.12 nan nan nan
    flow_out                    (nodes, techs, carriers, timesteps) float64 6kB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 6kB ...
    flow_export                 (nodes, techs, carriers, timesteps) float64 6kB ...
    source_use                  (nodes, techs, timesteps) float64 6kB nan ......
    ...                          ...
    min_cost_optimisation       float64 8B 20.36
    capacity_factor             (nodes, techs, carriers, timesteps) float64 6kB ...
    systemwide_capacity_factor  (techs, carriers) float64 40B 0.3351 ... 0.6978
    systemwide_levelised_cost   (techs, costs, carriers) float64 40B 0.001612...
    total_levelised_cost        (costs, carriers) float64 8B 0.002
    unmet_sum                   (nodes, carriers, timesteps) float64 1kB 0.0 ...

In [109]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes  techs              costs   
X1     X1_to_X2           monetary    7.042031
       X1_to_X3           monetary    1.731706
       pv                 monetary    0.000000
       supply_grid_power  monetary    2.639091
X2     X1_to_X2           monetary    7.042031
Name: cost, dtype: float64

In [110]:
lcoes = (
    model.results.systemwide_levelised_cost.sel(carriers="electricity")
    .to_series()
    .dropna()
)
lcoes.head()

techs              costs   
X1_to_X2           monetary    0.001612
X1_to_X3           monetary    0.003802
pv                 monetary    0.000423
supply_grid_power  monetary    0.000270
Name: systemwide_levelised_cost, dtype: float64

In [111]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

In [112]:
df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

      techs           timesteps  Flow in/out (kWh)
0  X1_to_X2 2005-07-01 00:00:00          -0.768974
1  X1_to_X2 2005-07-01 01:00:00          -0.625948
2  X1_to_X2 2005-07-01 02:00:00          -0.630137
3  X1_to_X2 2005-07-01 03:00:00          -0.630137
4  X1_to_X2 2005-07-01 04:00:00          -0.672884


In [113]:
carriers = ["electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

  nodes     techs     carriers           timesteps  Flow in/out (kWh)
0    X1  X1_to_X2  electricity 2005-07-01 00:00:00         -95.314775
1    X1  X1_to_X2  electricity 2005-07-01 01:00:00         -77.586567
2    X1  X1_to_X2  electricity 2005-07-01 02:00:00         -78.105887
3    X1  X1_to_X2  electricity 2005-07-01 03:00:00         -78.105887
4    X1  X1_to_X2  electricity 2005-07-01 04:00:00         -83.404380


In [114]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

  nodes              techs     carriers  Flow capacity (kW)
0    X1  supply_grid_power  electricity          291.454156
1    X2                 pv  electricity           44.544552
2    X3                 pv  electricity          250.000000


In [118]:
with open("model.yaml", "r", encoding="utf-8") as f:
    model_def = yaml.safe_load(f)

node_techs = {
    node: list(node_data.get("techs", {}).keys())
    for node, node_data in model_def.get("nodes", {}).items()
}

In [120]:
# Build a simple system map (nodes + links)
nodes = pd.read_csv("nodes_coordinates.csv")
links = pd.read_csv("links_techs.csv")

flow_cap = (
    model.results.flow_cap.to_series().dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)
flow_cap_lookup = dict(zip(flow_cap["techs"], flow_cap["Flow capacity (kW)"]))

center = [nodes.latitude.mean(), nodes.longitude.mean()]
system_map = folium.Map(location=center, zoom_start=15, tiles="CartoDB dark_matter")

# Add link lines
for _, row in links.iterrows():
    from_row = nodes.loc[nodes.nodes == row["link_from"]].iloc[0]
    to_row = nodes.loc[nodes.nodes == row["link_to"]].iloc[0]
    capacity = flow_cap_lookup.get(row["techs"], row.get("flow_cap_max"))
    popup = (
        f"<b>{row['techs']}</b><br>From: {row['link_from']}<br>To: {row['link_to']}"
    )
    if capacity is not None:
        popup += f"<br>Capacity: {capacity:.2f} kW"
    folium.PolyLine(
        locations=[[from_row.latitude, from_row.longitude], [to_row.latitude, to_row.longitude]],
        color=row.get("color", "#1f77b4"),
        weight=3,
        opacity=1,
        popup=popup,
    ).add_to(system_map)

# Add node markers
color_map = model.inputs.color.to_series().to_dict()
for _, row in nodes.iterrows():
    node = row["nodes"]
    techs = node_techs.get(node, [])
    base_types = (
        model.inputs.base_tech.sel(techs=techs).to_series().to_dict()
        if techs
        else {}
    )

    node_type = "Other"
    if any(t == "demand" for t in base_types.values()):
        node_type = "Demand"
    elif any(t == "supply" for t in base_types.values()):
        node_type = "Supply"

    marker_color = "#666666"
    for tech in techs:
        if tech in color_map:
            marker_color = color_map[tech]
            break

    popup = f"<b>{node}</b> ({node_type})<br>Techs: {', '.join(techs)}"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=marker_color,
        fill=True,
        fillColor=marker_color,
        fillOpacity=1,
        popup=popup,
    ).add_to(system_map)

system_map.save("system_map_simple.html")